# soft_prompt_ii_train_p668 — variant (ii), full training run at P = 668
Trains the soft prompt for variant (ii): P learnable rows appended to the MLLM's output, i.e. to the conditioning stream the diffusion transformer attends to. Everything else is frozen. OmniGen2 ships no soft-prompt recipe, so the loop is a patch onto `train.py`, applied with the same anchored `src.replace` pattern the LoRA notebooks use and validated by `soft_prompt_ii_probe_p668`.

Measured by the probe: 12.14 s/step at P=64, 15.06 s/step at P=668, peak 27.2 GB, and a usable learning rate of 1e-3 — roughly 1,250x LoRA's 8e-7. Under AdamW the per-coordinate step is set by the learning rate and is approximately invariant to gradient scale while sqrt(v_hat) >> eps, so this ratio is a genuine difference in distance travelled per step rather than an artefact of the two arms' loss normalisation. Learning rates at or above 1e-2 blow the prompt norm out by 2.5x to 22x while the gradient collapses by one to two orders of magnitude.

3000 steps, checkpoints written every 500. Writing a checkpoint is free (the prompt is a few MB); scoring one is not, so only four are scored later, at 500 / 1000 / 2000 / 3000. Selection is argmax of validation Overall over the scored checkpoints. If step 3000 takes the argmax, schedule a fresh run from step 0 with a longer budget rather than resuming: the prompt sits outside the accelerate state, so `prompt-3000.pt` cannot continue the trajectory under the current implementation.

KNOWN DEFECT IN THIS RUN. The prompt is appended after the collator's padding and the mask is extended with ones, so for the shorter row of a batch the mask is no longer a contiguous prefix. `OmniGen2RotaryPosEmbed.forward` reads `attention_mask.sum(dim=1)` and `OmniGen2Transformer2DModel.forward` slices `text_hidden_states[i, :cap_len]` from position 0, so that row consumes as many pad-position hidden states as it drops trailing prompt rows. Batch size is 2 and instruction lengths rarely tie, so one row per batch is affected. Variant (i) prepends and leaves the mask a prefix; variant (iii) does not change the sequence length; neither is affected. Generation is batch-of-1 and unpadded, so the evaluated prompt is injected cleanly. The defect is not confined to the trained artifact: this run's held-out diffusion validation loss evaluates at batch_size 2 through the same `_sp_encode` construction, so that curve inherits the same mask-prefix failure and is retained as a historical monitor rather than a clean conditioning-matched metric. The validation-loss inheritance adds nothing on top: checkpoint selection used GPT-4.1 validation Overall rather than diffusion loss, and generation is clean batch-of-1, so the selected step, the generated images and the test results carry no further effect from it. They remain outputs of a prompt trained under the defect, which a corrected run could in principle move. Not repeated: variant (iii) is correctly injected, parameter-matched at 100.3%, and lands within +0.023 Overall of this run (p 0.74). Variant (iii) is a different injection mechanism, so it does not bound the counterfactual corrected variant (ii), whose direction and magnitude on the final learned result are unknown. The deferral rests on cost and on (iii) already supplying a correctly implemented parameter-matched comparison.

This notebook is the **P = 668** run. Validation loss is logged every 250 steps on the same 98-row slice, timestep draws and seed the LoRA arm used. Not as an early-stopping signal — the LoRA arm established it runs opposite to the task metric — but as a replication test of that finding. This arm cannot carry that test on its own, since its validation encode path inherits the same defect as its training; the broader loss versus task-metric mismatch survives exclusion of this arm, though per-arm directions among the unaffected arms are mixed and it is not a per-arm replication.

## 1. Repo, deps

In [ ]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], cwd=REPO, check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-U","huggingface_hub[hf_xet]>=0.30.0,<1.0"], check=True)
    # requirements.txt omits torchdiffeq (omnigen2/transport/integrators.py) and peft (train.py)
    subprocess.run([sys.executable,"-m","pip","install","-q","torchdiffeq"], check=True)
    print("installed.")
else:
    print("OmniGen2 already present.")

# peft's is_torchao_available() raises rather than returning False when torchao predates 0.16;
# Colab ships 0.10. torchao is unused by this bf16 path. Guarded, so a no-op once gone.
_ao = subprocess.run([sys.executable,"-c",
                      "from peft.import_utils import is_torchao_available; is_torchao_available()"],
                     capture_output=True, text=True)
if _ao.returncode != 0:
    subprocess.run([sys.executable,"-m","pip","uninstall","-y","-q","torchao"], check=True)
    print("torchao removed")
sys.path.insert(0, REPO); os.chdir(REPO)
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())

installed.
torchao removed
NVIDIA A100-SXM4-80GB, 81920 MiB


## 2. Drive, images, splits

In [ ]:
from google.colab import drive, userdata
import json
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
IMGS, EXPECTED = "/content/imgs", 37063
TAR = f"{DRIVE}/imgs_subset.tar"

def n_files(d):
    return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}',
                   shell=True, check=True)
    print("restored", n_files(IMGS), "/", EXPECTED)
    assert n_files(IMGS) >= EXPECTED, "restore incomplete"
else:
    print("images complete:", n_files(IMGS))

for s in ["train", "val"]:
    src = f"{DRIVE}/{s}.jsonl"
    if not os.path.exists(f"/content/{s}.jsonl"):
        subprocess.run(["cp", src, f"/content/{s}.jsonl"], check=True)
subprocess.run(["cp", f"{DRIVE}/screen_exclude_from_training.txt", "/content/"], check=True)
print("splits restored")

Mounted at /content/drive
restored 37063 / 37063
splits restored


## 3. Prompt length, initialisation, run identity
`P` is the number of learnable rows; each is 2048 wide, the MLLM's hidden size, so trainable parameters are `P x 2048`.

`P = 64` is not run: the probe measured the same +3.6% drift at 64 and at 668, so it cannot distinguish anything, and at 5% of rank 4 it is the size furthest from the parameter-matching requirement. `P = 668` keeps instruction plus prompt inside the 888-token length the decoder was trained on for 96.23% of training rows (38,098 of 39,591), measured with the model's own tokenizer on templated text: the longest training instruction is 263 tokens raw and 288 templated at a constant 25-token overhead, so the longest stream reaches 956 conditioning rows and 1,493 rows exceed 888, and is the parameter-comparable point at 54% of LoRA rank 4. The nearest-integer shallow match is 1,247 rows, at 100.01% of rank 4; nothing architectural prevents it once the RoPE table is extended, and it is rejected on design grounds because it would push the conditioning stream to 1,535 rows against the 888 seen in training. Separately from that 888-token training length, the released rotary tables cap the joint text axis at 1024, so both P values require extending them; the extension is exact because get_1d_rotary_pos_embed row k is identical for any table length above k.

Rows are initialised from the embeddings of real domain words rather than noise. Lester et al. (2021) evaluate sampled-vocabulary initialisation alongside random-uniform and class-label initialisation; no alternative initialisation is run here, so this project makes no claim about its effect.

In [ ]:
import torch, re
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import AutoTokenizer

P    = 668       # this notebook is the P=668 run
LR   = 1e-3      # from soft_prompt_ii_probe_p668, chosen on prompt drift; the gradient norm is the
                 # supporting check — it peaks here and collapses one to two orders above
STEPS = 3000
CKPT_EVERY = 500
SEED = 2233
MODEL = "OmniGen2/OmniGen2"
RUN = f"sp_ii_p{P}"

torch.manual_seed(SEED)
tok = AutoTokenizer.from_pretrained(MODEL, subfolder="processor", trust_remote_code=True)

# read only the embedding matrix out of the mllm shards, not the whole 3B model
idx = json.load(open(hf_hub_download(MODEL, "mllm/model.safetensors.index.json")))
key = next(k for k in idx["weight_map"] if k.endswith("embed_tokens.weight"))
emb = load_file(hf_hub_download(MODEL, f"mllm/{idx['weight_map'][key]}"))[key]

SEED_WORDS = ("a photo of a laptop tablet smartphone headphones earbuds keyboard computer mouse "
              "game controller gaming console desktop computer setup vr headset on a wooden desk "
              "in an office studio lighting product shot detailed realistic")
ids  = tok(SEED_WORDS, add_special_tokens=False).input_ids
init = emb[torch.tensor((ids * ((P // len(ids)) + 1))[:P])].clone().float()
assert init.shape == (P, emb.shape[1]), init.shape

EXP = f"{DRIVE}/soft_prompt/experiments/{RUN}"
os.makedirs(EXP, exist_ok=True)
os.makedirs("/content/sp", exist_ok=True)
torch.save(init, "/content/sp/init.pt")
json.dump({"P": P, "dim": int(emb.shape[1]), "init_path": "/content/sp/init.pt",
           "seed": SEED, "save_dir": EXP, "every": CKPT_EVERY},
          open("/content/soft_prompt.json", "w"))

LORA_R4 = 38 * 16800 * 4
print(f"run {RUN}  P={P}  dim={emb.shape[1]}  trainable={init.numel():,} "
      f"= {100*init.numel()/LORA_R4:.0f}% of LoRA r4")
print(f"lr {LR}  steps {STEPS}  checkpoints every {CKPT_EVERY} -> {EXP}")
print(f"init norm {init.norm():.3f}  (the probe measured 20.610 at this P)")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

processor/tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

mllm/model-00001-of-00004.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

run sp_ii_p668  P=668  dim=2048  trainable=1,368,064 = 54% of LoRA r4
lr 0.001  steps 3000  checkpoints every 500 -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/soft_prompt/experiments/sp_ii_p668
init norm 20.611  (the probe measured 20.610 at this P)


## 4. Training data
The full domain training split with the 62 screen targets excluded, in OmniGen2's format. Identical row set to the LoRA arm.

The length diagnostic below counts the raw instruction and is superseded. It reports 263 tokens, a 931-row stream and 99.39% inside 888. The collator truncates the chat-templated string, not the raw one, at a constant 25-token overhead measured in `inspect_template_surface`: the maximum templated instruction is 288, the P=668 stream reaches 956, and 38,098 of 39,591 training rows (96.23%) sit at or below 888 with 1,493 above. On the frozen splits the maxima are 913 on test (14 rows above 888) and 892 on val (2 rows above). The output is retained as executed; the counts are print-only and feed neither the data nor the optimiser.

In [ ]:
rows = [json.loads(l) for l in open("/content/train.jsonl")]
excl = {l.strip() for l in open("/content/screen_exclude_from_training.txt") if l.strip()}
rows = [r for r in rows if r["output_image"] not in excl]
assert len(rows) == 39591, f"expected 39591 training rows, got {len(rows)}"
from transformers import AutoTokenizer
_tok = AutoTokenizer.from_pretrained(MODEL, subfolder="processor", trust_remote_code=True)
_len = [len(x) for x in _tok([r["instruction"] for r in rows], add_special_tokens=False).input_ids]
MAX_INSTR = max(_len)
print(f"longest training instruction {MAX_INSTR} tokens; prompt {P} + {MAX_INSTR} = {P + MAX_INSTR} conditioning rows")
print(f"{100*sum(1 for n in _len if P + n <= 888)/len(_len):.2f}% of training rows keep prompt+instruction inside the 888 seen in training")
# not a hard cap for variant (ii): the prefix is concatenated after the collator truncates the instruction, so this reports distribution, not a limit
print(f"train rows {len(rows)}  (39,893 minus the 302 screen rows)")

os.makedirs("/content/data", exist_ok=True)
with open("/content/data/train_sp.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps({
            "task_type": "ic",
            "instruction": r["instruction"],
            "input_images": [os.path.join(IMGS, p) for p in r["input_images"]],
            "output_image": os.path.join(IMGS, r["output_image"]),
        }) + "\n")

os.makedirs(f"{REPO}/data_configs/train/sp", exist_ok=True)
with open(f"{REPO}/data_configs/train/sp/sp.yml", "w") as f:
    f.write("data:\n  -\n    path: /content/data/train_sp.jsonl\n    type: 'ic'\n    ratio: !!float 1\n")
print("data config written")

longest training instruction 263 tokens; prompt 668 + 263 = 931 conditioning rows
99.39% of training rows keep prompt+instruction inside the 888 seen in training
train rows 39591  (39,893 minus the 302 screen rows)
data config written


## 5. Patch train.py — freeze everything, train the prompt only
The two edits validated by the probe, each asserted to match exactly once.

**A** replaces the optimizer's parameter list: the transformer, VAE and text encoder are frozen and the optimizer is given the prompt alone. `accelerator.clip_grad_norm_` at line 592 already takes `trainable_params`, so clipping follows.

**B** appends the prompt to `text_feats` immediately after the `torch.no_grad()` block that produces it, and extends `text_attention_mask` to match. Placement after the block is what makes the prompt trainable: the encoder output is detached, the prompt is not. The probe confirmed a non-zero gradient reaches it.

In [ ]:
p = f"{REPO}/train.py"
src = open(p, encoding="utf-8").read()

A = "    trainable_params = list(filter(lambda p: p.requires_grad, model.parameters()))\n"
A_NEW = "\n".join([
    "    # soft prompting variant (ii): the only trainable tensor is a P x hidden prompt appended to",
    "    # the MLLM output. Transformer, VAE and text encoder are frozen. OmniGen2 ships no soft-prompt",
    "    # path, so this replaces the LoRA/full-FT parameter selection rather than extending it.",
    "    import json as _json",
    "    _spc = _json.load(open('/content/soft_prompt.json'))",
    "    model.requires_grad_(False)",
    "    SOFT_PROMPT = torch.nn.Parameter(",
    "        torch.load(_spc['init_path'], map_location='cpu').to(accelerator.device, dtype=torch.float32))",
    "    SOFT_PROMPT.requires_grad_(True)",
    "    globals()['SOFT_PROMPT'] = SOFT_PROMPT",
    "    globals()['SP_CFG'] = _spc",
    "    trainable_params = [SOFT_PROMPT]",
    "    logger.info('[soft_prompt] P=%d dim=%d trainable=%d' % (",
    "        SOFT_PROMPT.shape[0], SOFT_PROMPT.shape[1], SOFT_PROMPT.numel()))",
]) + "\n"

B = "                    ).last_hidden_state\n"
B_NEW = "\n".join([
    "                # appended OUTSIDE the no_grad block above: text_feats is detached, SOFT_PROMPT is",
    "                # not, so the loss reaches the prompt through the diffusion transformer",
    "                _sp = SOFT_PROMPT.to(text_feats.dtype).unsqueeze(0).expand(text_feats.shape[0], -1, -1)",
    "                text_feats = torch.cat([text_feats, _sp], dim=1)",
    "                text_mask = torch.cat([text_mask,",
    "                    torch.ones(text_mask.shape[0], _sp.shape[1],",
    "                               dtype=text_mask.dtype, device=text_mask.device)], dim=1)",
    "                if not text_feats.requires_grad:",
    "                    raise RuntimeError('soft prompt is not in the graph - check patch B placement')",
]) + "\n"

if "[soft_prompt]" in src:
    print("already patched")
else:
    assert src.count(A) == 1, f"anchor A found {src.count(A)} times"
    assert src.count(B) == 1, f"anchor B found {src.count(B)} times"
    open(p, "w", encoding="utf-8").write(src.replace(A, A_NEW).replace(B, B + B_NEW))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("patched and parses")

# repo.py writes text positions into all three rope axes, and the tables are sized from the
# released config. With P appended to the conditioning stream, cap_seq_len is real_text_len + P,
# which indexes past the table on the longest rows. get_1d_rotary_pos_embed is outer(arange(e),
# freqs), so row k is identical for any e > k: enlarging the table is a pure extension.
# The released config bounds axis 0 at 1024: P=668 exceeds it on the longest rows, and the
# parameter-matched P=1247 would exceed it on every row. Extending the table lifts that bound;
# the remaining concern at P=1247 is conditioning length against the 888 rows seen in training.
ROPE_LEN = 4096   # covers the 888 collator cap plus the largest P under test

R = """    freqs_cis = OmniGen2RotaryPosEmbed.get_freqs_cis(
        model.config.axes_dim_rope,
        model.config.axes_lens,
        theta=10000,
    )
"""
R_NEW = f"""    _axes_lens = tuple(max(int(l), {ROPE_LEN}) for l in model.config.axes_lens)
    logger.info("[soft_prompt] rope axes_lens %s -> %s" % (tuple(model.config.axes_lens), _axes_lens))
    freqs_cis = OmniGen2RotaryPosEmbed.get_freqs_cis(
        model.config.axes_dim_rope,
        _axes_lens,
        theta=10000,
    )
"""

src = open(p, encoding="utf-8").read()
if "_axes_lens" in src:
    print("rope tables already extended")
else:
    assert src.count(R) == 1, "rope anchor R not found exactly once"
    src = src.replace(R, R_NEW)
    ast.parse(src)
    open(p, "w", encoding="utf-8").write(src)
    print(f"rope tables extended to {ROPE_LEN}")

patched and parses
rope tables extended to 4096


## 5a. Loss normalisation
gradient_state.num_steps is 1 on the reference 8-GPU config and 8 here (single GPU, accum 8),
so the shipped normalisation runs at 8x the reference gradient scale. Corrected to match the
LoRA arm, which applies the same replacement. Derivation in lora_train_r{4,8,16} section 5.

In [ ]:
FIX_ACCUM_NORM = True

p = f"{REPO}/train.py"
A4     = ("                loss = (loss * accelerator.gradient_state.num_steps"
          " * accelerator.num_processes) / num_tokens_in_batch\n")
A4_NEW = "                loss = (loss * accelerator.num_processes) / num_tokens_in_batch\n"

src = open(p, encoding="utf-8").read()
if A4_NEW in src:
    print("accum-norm already corrected")
elif not FIX_ACCUM_NORM:
    print("FIX_ACCUM_NORM=False - shipped normalisation retained (8x reference gradient scale)")
else:
    assert src.count(A4) == 1, f"anchor A4 found {src.count(A4)} times"
    open(p, "w", encoding="utf-8").write(src.replace(A4, A4_NEW))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("accum-norm corrected and train.py parses")

now = open(p, encoding="utf-8").read()
print(f"  accum-norm fix : {A4_NEW in now}  (FIX_ACCUM_NORM={FIX_ACCUM_NORM})")
print(f"  shipped line   : {A4 in now}")

accum-norm corrected and train.py parses
  accum-norm fix : True  (FIX_ACCUM_NORM=True)
  shipped line   : False


## 6. Checkpointing and monitor
`accelerator.save_state` captures the model, and the prompt is deliberately not part of it, so **S** writes the prompt itself at every checkpoint interval. Each file is a few MB.

**M** prints the gradient norm and the prompt norm every 50 steps. Over a 12-hour run this is the live answer to whether the arm is learning: a gradient that dies or a prompt norm that stops moving flags optimisation saturation or a stall, to be read against the loss curve rather than taken as an expressiveness limit. The probe's readings at lr 1e-3 and this P were gradient 6.184e-03 and prompt norm rising 20.610 -> 21.346 over 15 steps.

In [ ]:
src = open(p, encoding="utf-8").read()

S = "                if global_step % args.logger.checkpointing_steps == 0:\n"
S_NEW = "\n".join([
    "                if global_step % SP_CFG['every'] == 0 and accelerator.is_main_process:",
    "                    import os as _os",
    "                    _d = SP_CFG['save_dir']",
    "                    _os.makedirs(_d, exist_ok=True)",
    "                    torch.save({'soft_prompt': SOFT_PROMPT.detach().cpu(),",
    "                                'step': global_step, 'P': SOFT_PROMPT.shape[0],",
    "                                'dim': SOFT_PROMPT.shape[1]},",
    "                               _os.path.join(_d, 'prompt-%d.pt' % global_step))",
    "                    logger.info('[sp] saved prompt-%d.pt' % global_step)",
]) + "\n"

M = "                    accelerator.clip_grad_norm_(trainable_params, args.train.max_grad_norm)\n"
M_NEW = "\n".join([
    # clip_grad_norm_ returns the PRE-clip total norm; reading .grad after the call
    # returns the clipped value, which saturates at max_grad_norm and is not a measurement
    "                    _gnorm = accelerator.clip_grad_norm_(trainable_params, args.train.max_grad_norm)",
    "                    if accelerator.is_main_process and global_step % 50 == 0:",
    "                        print('[sp] step %d grad_norm %.3e prompt_norm %.4f' % (",
    "                            global_step, float(_gnorm),",
    "                            SOFT_PROMPT.detach().float().norm().item()), flush=True)",
]) + "\n"

if "[sp] saved" in src:
    print("already patched")
else:
    assert src.count(S) == 1, f"anchor S found {src.count(S)} times"
    assert src.count(M) == 1, f"anchor M found {src.count(M)} times"
    open(p, "w", encoding="utf-8").write(src.replace(S, S_NEW + S).replace(M, M_NEW))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("checkpoint and monitor patched")

checkpoint and monitor patched


## 7. Validation loss module
The evaluator the LoRA arm used, plus an `encode_text` hook. Without it the val path would call the frozen text encoder directly and measure the model with no prompt attached — a flat curve that looks like a result.

In [ ]:
VAL_MODULE = '"""Held-out validation loss for OmniGen2 training.\n\nEvaluates a deterministic held-out proxy for the training objective, using the VAE posterior mode rather than a posterior sample so successive evaluations isolate weight changes, on a held-out split with gradients disabled.\nTimesteps and noise are drawn through the repository\'s own transport.sample under\na fixed seed, so the timestep distribution and resolution-dependent shift match\ntraining exactly while successive evaluations differ only by the model weights.\n\nGlobal RNG state is saved and restored around every evaluation so that the\ntraining stream is unaffected.\n\nNot part of the upstream OmniGen2 repository.\n"""\n\nimport json\nimport os\nimport time\n\nimport torch\n\n\nclass ValLossEvaluator:\n    """Per-token velocity MSE on a held-out split at fixed timesteps and noise."""\n\n    def __init__(self, dataset, collator, transport, out_path, batch_size=2,\n                 every=250, n_repeats=4, seed=1234, encode_text=None):\n        self.dataset = dataset\n        self.collator = collator\n        self.transport = transport\n        self.out_path = out_path\n        self.batch_size = batch_size\n        self.every = every\n        self.n_repeats = n_repeats\n        self.seed = seed\n        self.encode_text = encode_text\n        d = os.path.dirname(out_path)\n        if d:\n            os.makedirs(d, exist_ok=True)\n\n    @torch.no_grad()\n    def run(self, model, vae, text_encoder, freqs_cis, weight_dtype, global_step, device):\n        t_start = time.time()\n        was_training = model.training\n        model.eval()\n\n        cpu_rng = torch.get_rng_state()\n        cuda_rng = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None\n\n        sse = 0.0\n        tokens = 0.0\n        by_t = []\n\n        try:\n            for start in range(0, len(self.dataset), self.batch_size):\n                idx = list(range(start, min(start + self.batch_size, len(self.dataset))))\n                batch = self.collator([self.dataset[i] for i in idx])\n\n                text_ids = batch["text_ids"].to(device)\n                text_mask = batch["text_mask"].to(device)\n                if self.encode_text is None:\n                    text_feats = text_encoder(\n                        input_ids=text_ids,\n                        attention_mask=text_mask,\n                        output_hidden_states=False,\n                    ).last_hidden_state\n                else:\n                    # soft-prompt arms: mirror the training-loop injection so val loss is\n                    # measured on the same conditioning the optimiser is shaping\n                    text_feats, text_mask = self.encode_text(text_encoder, text_ids, text_mask)\n\n                def encode(img):\n                    z = vae.encode(img.to(device=device, dtype=vae.dtype)).latent_dist.mode()\n                    if vae.config.shift_factor is not None:\n                        z = z - vae.config.shift_factor\n                    if vae.config.scaling_factor is not None:\n                        z = z * vae.config.scaling_factor\n                    return z.to(dtype=weight_dtype)\n\n                ref_latents = []\n                for imgs in batch["input_images"]:\n                    if imgs is not None and len(imgs) > 0:\n                        ref_latents.append([encode(im).squeeze(0) for im in imgs])\n                    else:\n                        ref_latents.append(None)\n\n                x1 = [encode(im).squeeze(0) for im in batch["output_image"]]\n\n                model_kwargs = dict(\n                    text_hidden_states=text_feats,\n                    text_attention_mask=text_mask,\n                    ref_image_hidden_states=ref_latents,\n                    freqs_cis=freqs_cis,\n                )\n\n                for k in range(self.n_repeats):\n                    torch.manual_seed((self.seed * 100003 + start * 101 + k) % (2 ** 31 - 1))\n                    t, x0, _ = self.transport.sample(x1, 0, 1)\n                    t, xt, ut = self.transport.path_sampler.plan(t, x0, x1)\n                    pred = model(xt, t, **dict(model_kwargs))\n                    for j in range(len(x1)):\n                        d = (pred[j].float() - ut[j].float()) ** 2\n                        sse += float(d.sum().item())\n                        tokens += float(d.numel())\n                        by_t.append((float(t[j].item()), float(d.mean().item())))\n        finally:\n            torch.set_rng_state(cpu_rng)\n            if cuda_rng is not None:\n                torch.cuda.set_rng_state_all(cuda_rng)\n            if was_training:\n                model.train()\n\n        val_loss = sse / max(tokens, 1.0)\n\n        bins = [0.0] * 4\n        counts = [0] * 4\n        for t_val, mse in by_t:\n            b = min(int(t_val * 4), 3)\n            bins[b] += mse\n            counts[b] += 1\n        binned = {f"t_{i/4:.2f}_{(i+1)/4:.2f}": (bins[i] / counts[i] if counts[i] else None)\n                  for i in range(4)}\n\n        with open(self.out_path, "a", encoding="utf-8") as f:\n            f.write(json.dumps({\n                "step": int(global_step),\n                "val_loss": val_loss,\n                "n_rows": len(self.dataset),\n                "n_repeats": self.n_repeats,\n                "n_samples": len(by_t),\n                "seconds": round(time.time() - t_start, 1),\n                "by_timestep_quartile": binned,\n            }) + "\\n")\n\n        return val_loss\n'

path = f"{REPO}/omnigen2/val_loss.py"
open(path, "w", encoding="utf-8").write(VAL_MODULE)
import ast; ast.parse(open(path, encoding='utf-8').read())
print(f"wrote {path} ({len(VAL_MODULE)} chars) and it parses")

wrote /content/OmniGen2/omnigen2/val_loss.py (5257 chars) and it parses


## 8. Validation loss patch
The 98-row val slice, written in the same form as the training pool, and the two anchors that install the evaluator. `_sp_encode` mirrors patch B exactly, so validation loss is measured on the conditioning the optimiser is actually shaping.

In [ ]:
# validation loss on the frozen 98-row val slice: same rows, timestep draws and seed
# as the LoRA arm. The conditioning path is not matched: this arm's _sp_encode carries the
# append-after-padding defect at batch_size 2, so the curve is a historical monitor rather
# than a clean cross-method comparison
val_rows = [json.loads(l) for l in open("/content/val.jsonl", encoding="utf-8")]
assert len(val_rows) == 98, f"expected 98 val rows, got {len(val_rows)}"
assert not ({r["output_image"] for r in rows} & {r["output_image"] for r in val_rows}), \
    "val targets leaked into training"

with open("/content/data/val_sp.jsonl", "w", encoding="utf-8") as f:
    for r in val_rows:
        f.write(json.dumps({
            "task_type": "ic",
            "instruction": r["instruction"],
            "input_images": [os.path.join(IMGS, p) for p in r["input_images"]],
            "output_image": os.path.join(IMGS, r["output_image"]),
        }) + "\n")
with open(f"{REPO}/data_configs/train/sp/val_sp.yml", "w") as f:
    f.write("data:\n  -\n    path: /content/data/val_sp.jsonl\n    type: 'ic'\n    ratio: !!float 1\n")
print(f"val config written: {len(val_rows)} rows")

src = open(p, encoding="utf-8").read()
A1 = '    logger.info(f"{args.train.batch_size=} {args.train.gradient_accumulation_steps=} {accelerator.num_processes=} {args.train.global_batch_size=}")\n'
A2 = "                global_step += 1\n"

SETUP = '''    # held-out validation loss (not part of upstream OmniGen2)
    _val_eval = None
    if args.get("val_loss", None) is not None and accelerator.is_main_process:
        from omnigen2.val_loss import ValLossEvaluator
        _val_dataset = OmniGen2TrainDataset(
            args.val_loss.data_path,
            tokenizer=text_tokenizer,
            use_chat_template=args.data.use_chat_template,
            prompt_dropout_prob=0.0,
            ref_img_dropout_prob=0.0,
            max_input_pixels=OmegaConf.to_object(args.data.get("max_input_pixels", 1024 * 1024)),
            max_output_pixels=args.data.get("max_output_pixels", 1024 * 1024),
            max_side_length=args.data.get("max_side_length", 2048),
        )
        # variant (ii): prompt appended to the MLLM output
        def _sp_encode(enc, ids, mask):
            _f = enc(input_ids=ids, attention_mask=mask,
                     output_hidden_states=False).last_hidden_state
            _sp = SOFT_PROMPT.to(_f.dtype).unsqueeze(0).expand(_f.shape[0], -1, -1)
            _mask = torch.cat([mask, torch.ones(mask.shape[0], _sp.shape[1],
                                                dtype=mask.dtype, device=mask.device)], dim=1)
            return torch.cat([_f, _sp], dim=1), _mask
        _val_eval = ValLossEvaluator(
            dataset=_val_dataset,
            collator=OmniGen2Collator(tokenizer=text_tokenizer,
                                      max_token_len=args.data.maximum_text_tokens),
            transport=transport,
            out_path=args.val_loss.out_path,
            batch_size=args.val_loss.get("batch_size", 2),
            every=args.val_loss.get("every", 250),
            n_repeats=args.val_loss.get("n_repeats", 4),
            seed=args.val_loss.get("seed", 1234),
            encode_text=_sp_encode,
        )
        logger.info(f"[val] {len(_val_dataset)} rows every {_val_eval.every} steps, "
                    f"{_val_eval.n_repeats} draws per row")

'''

CALL = '''                if _val_eval is not None and global_step % _val_eval.every == 0:
                    try:
                        _vl = _val_eval.run(model, vae, text_encoder, freqs_cis,
                                            weight_dtype, global_step, accelerator.device)
                        logs["val_loss"] = _vl
                        logger.info("[val] step %d val_loss %.6f" % (global_step, _vl))
                    except Exception as _e:
                        logger.warning("[val] step %d failed: %r" % (global_step, _e))
'''

if "_val_eval" in src:
    print("val hook already patched")
else:
    assert src.count(A1) == 1, f"anchor A1 found {src.count(A1)} times"
    assert src.count(A2) == 1, f"anchor A2 found {src.count(A2)} times"
    open(p, "w", encoding="utf-8").write(src.replace(A1, SETUP + A1).replace(A2, A2 + CALL))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("val hook patched")

val config written: 98 rows
val hook patched


## 9. Config
`ft_lora.yml` with `lora_ft` off, the three resolution keys at 512, and `gradient_accumulation_steps: 8` so `global_batch_size: 16` holds on one GPU — the same deviations the LoRA runs record. `warmup_t` is back to the shipped 500, unlike the probe, so the schedule shape matches the LoRA arm. `checkpointing_steps` is set beyond the run length and patch S inserts the prompt write above the shipped checkpoint block rather than replacing its condition, so `accelerator.save_state` stayed gated on `checkpointing_steps` and never fired: no accelerate state directories were written. The prompt tensors patch S writes are the checkpoints this arm evaluates.

In [ ]:
import yaml
cfg = yaml.safe_load(open(f"{REPO}/options/ft_lora.yml"))
cfg["train"]["lora_ft"] = False
cfg["train"]["batch_size"] = 2
cfg["train"]["gradient_accumulation_steps"] = 8
cfg["train"]["global_batch_size"] = 16
cfg["train"]["max_train_steps"] = STEPS
cfg["train"]["learning_rate"] = LR
cfg["train"]["gradient_checkpointing"] = True
cfg["name"] = RUN     # train.py:81 derives output_dir from conf.name; there is no train.output_dir
cfg["logger"]["log_with"] = ["tensorboard"]        # wandb would block on an interactive login
cfg["logger"]["checkpointing_steps"] = STEPS * 10  # gates the shipped save_state block out of
                                                   # a 3000-step run; patch S writes the prompt itself
cfg["data"]["max_output_pixels"] = 512*512
cfg["data"]["max_input_pixels"] = [512*512]*4
cfg["data"]["max_side_length"] = 512
cfg["data"]["data_path"] = "data_configs/train/sp/sp.yml"
cfg["val_loss"] = {                                # 98 rows, 4 fixed-seed draws, as the LoRA arm
    "data_path": "data_configs/train/sp/val_sp.yml",
    "out_path": f"{EXP}/val_loss.jsonl",
    "every": 250,
    "batch_size": 2,
    "n_repeats": 4,
    "seed": 1234,
}

OPT = f"/content/{RUN}.yml"
yaml.safe_dump(cfg, open(OPT, "w"), sort_keys=False)
print(f"lr {cfg['train']['learning_rate']} | steps {cfg['train']['max_train_steps']} | "
      f"warmup_t {cfg['train']['warmup_t']} | maximum_text_tokens {cfg['data']['maximum_text_tokens']}")
print(f"batch {cfg['train']['batch_size']} x accum {cfg['train']['gradient_accumulation_steps']} "
      f"= global {cfg['train']['global_batch_size']}")
print(f"val loss every {cfg['val_loss']['every']} steps -> {cfg['val_loss']['out_path']}")

lr 0.001 | steps 3000 | warmup_t 500 | maximum_text_tokens 888
batch 2 x accum 8 = global 16
val loss every 250 steps -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/soft_prompt/experiments/sp_ii_p668/val_loss.jsonl


## 10. Train
NOT resumable. Patch S inserts the prompt write above the shipped checkpoint block and leaves its condition alone, so `accelerator.save_state` stayed gated on `checkpointing_steps` and never fired. It would not have helped — the prompt is a bare Parameter outside `model`, so `save_state` does not capture it, and `resume_from_checkpoint: latest` would restore weights, optimizer moments and the step counter onto a prompt reset to init. This run went 0 to 3000 in a single session, logged steps advancing by 50 with no gap, so no resume occurred. Patch S is the mitigation: a run that dies at 2000 still leaves four prompts, three of them scoring points. Expect roughly 12.6 h from the probe's measured 15.06 s/step at P=668.

In [ ]:
_s = open(p, encoding='utf-8').read()
for _k in ('[soft_prompt]', '[sp] saved', '[sp] step', '_val_eval', '_axes_lens'):
    assert _k in _s, f'train.py is missing patch marker {_k!r} - re-run sections 5 to 8'
assert 'loss = (loss * accelerator.num_processes) / num_tokens_in_batch' in _s, \
    'train.py loss normalisation not corrected - re-run section 5a'
print('all patches present in train.py')
import time, re
t0 = time.time()
proc = subprocess.Popen(["accelerate","launch","--num_processes=1","--mixed_precision=bf16",
                         "train.py","--config",OPT],
                        cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env={**os.environ, "WANDB_DISABLED": "true"})
LOG = f"/content/{RUN}_console.log"
with open(LOG, "w") as fh:
    for line in proc.stdout:
        fh.write(line)
        if re.search(r"\[sp\]|\[soft_prompt\]|Error|Traceback|error", line):
            print(f"[{(time.time()-t0)/3600:5.2f}h] {line.rstrip()[:150]}", flush=True)
proc.wait()
print(f"\nexit {proc.returncode} after {(time.time()-t0)/3600:.2f}h")
subprocess.run(["cp", LOG, f"{EXP}/console.log"])

all patches present in train.py
[ 0.00h] 2026-08-01 18:57:24.797751: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due 
[ 0.02h] 08/01/2026 18:58:19 - INFO - __main__ - [RANK 0] [soft_prompt] rope axes_lens (1024, 1664, 1664) -> (4096, 4096, 4096)
[ 0.03h] 08/01/2026 18:58:59 - INFO - __main__ - [RANK 0] [soft_prompt] P=668 dim=2048 trainable=1368064
[ 0.04h] [sp] step 0 grad_norm 1.929e-02 prompt_norm 20.6103
[ 0.24h] [sp] step 50 grad_norm 8.895e-03 prompt_norm 20.6229
[ 0.45h] [sp] step 100 grad_norm 1.739e-02 prompt_norm 20.6715
[ 0.65h] [sp] step 150 grad_norm 2.419e-02 prompt_norm 20.7713
[ 0.86h] [sp] step 200 grad_norm 1.282e-02 prompt_norm 20.9374
[ 1.10h] [sp] step 250 grad_norm 2.791e-02 prompt_norm 21.2235
[ 1.31h] [sp] step 300 grad_norm 1.301e-02 prompt_norm 21.6086
[ 1.51h] [sp] step 350 grad_norm 6.105e-03 prompt_norm 22.2250
[ 1.72h] [sp] step 400 grad_norm 1.476e-02 prompt_norm 22.7996
[ 1.92h] [s

CompletedProcess(args=['cp', '/content/sp_ii_p668_console.log', "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/soft_prompt/experiments/sp_ii_p668/console.log"], returncode=0)

## 11. Checkpoint inventory

In [ ]:
import glob
files = sorted(glob.glob(f"{EXP}/prompt-*.pt"), key=lambda f: int(re.search(r"prompt-(\d+)", f).group(1)))
print(f"{'step':>6s} {'MB':>7s} {'prompt norm':>12s} {'drift from init':>16s}")
init_n = init.norm().item()
for f in files:
    d = torch.load(f, map_location="cpu")
    n = d["soft_prompt"].float().norm().item()
    print(f"{d['step']:>6d} {os.path.getsize(f)/1e6:>7.2f} {n:>12.4f} {100*(n-init_n)/init_n:>15.1f}%")
print(f"{len(files)} of {STEPS//CKPT_EVERY} expected checkpoints present")
json.dump({"run": RUN, "P": P, "lr": LR, "steps": STEPS, "seed": SEED,
           "trainable": int(init.numel()), "checkpoints": [int(re.search(r"prompt-(\d+)", f).group(1)) for f in files],
           "train_rows": len(rows)}, open(f"{EXP}/manifest.json", "w"), indent=1)
print(f"\nmanifest written to {EXP}/manifest.json")

  step      MB  prompt norm  drift from init
   500    5.47      24.7024            19.8%
  1000    5.47      34.5548            67.6%
  1500    5.47      42.3038           105.2%
  2000    5.47      48.9343           137.4%
  2500    5.47      54.6995           165.4%
  3000    5.47      59.8068           190.2%
6 of 6 expected checkpoints present

manifest written to /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/soft_prompt/experiments/sp_ii_p668/manifest.json


## 12. Revert the patch

In [ ]:
subprocess.run(["git","-C",REPO,"checkout","--","train.py"], check=True)
s = open(f"{REPO}/train.py", encoding="utf-8").read()
assert "[soft_prompt]" not in s and "[sp] saved" not in s
assert "loss * accelerator.gradient_state.num_steps" in s, "loss normalisation not restored"
print("train.py reverted to HEAD")

train.py reverted to HEAD
